In [1]:
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = "google/gemma-4-E2B-it"
# Load model
processor = AutoProcessor.from_pretrained(MODEL_ID)



chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Tokenization · approver check
#  HF processor is ground truth; our wrapper must match.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import torch
from architecture.tokenization import GemmaTokenizer

tok = GemmaTokenizer(processor)

s = "Explain MoE in transformers in 3 sentences."

ours  = tok.encode(s)[0]
their = processor.tokenizer(s, return_tensors="pt")["input_ids"][0]

print(f"vocab_size = {tok.vocab_size:,}")
print(f"ours  ({ours.shape[0]}): {ours.tolist()[:10]} ...")
print(f"their ({their.shape[0]}): {their.tolist()[:10]} ...")
assert torch.equal(ours, their)
print("✓ match")

print("\nfirst 10 pieces:")
for tid, piece in tok.pretty(ours)[:10]:
    print(f"  {tid:>6}  {piece!r}")

print(f"\nroundtrip: {tok.decode(ours)!r}")


vocab_size = 262,144
ours  (10): [155122, 7945, 236788, 528, 39687, 528, 236743, 236800, 23974, 236761] ...
their (10): [155122, 7945, 236788, 528, 39687, 528, 236743, 236800, 23974, 236761] ...
✓ match

first 10 pieces:
  155122  'Explain'
    7945  '▁Mo'
  236788  'E'
     528  '▁in'
   39687  '▁transformers'
     528  '▁in'
  236743  '▁'
  236800  '3'
   23974  '▁sentences'
  236761  '.'

roundtrip: 'Explain MoE in transformers in 3 sentences.'


In [ ]:
# load the actual model to make sure it works
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)